# Beyond Visual Explanations — Final Experiment Notebook

Pneumonia detection on the Kaggle chest X-ray dataset (Kermany et al.) with **ResNet50** and **DenseNet121**,
explained with **Grad-CAM, LIME and Integrated Gradients**, and evaluated quantitatively.

**What this notebook produces (all saved in `results/`):**
1. Two fine-tuned models with a proper validation split (10 % of training data, stratified).
2. Test-set classification metrics, confusion matrices, ROC curves, learning curves.
3. Deletion / insertion AUC for **all 6 model–XAI pairs** on the test images, plus a random-attribution control.
4. Statistics: mean ± SE, bootstrap 95 % CIs, paired Wilcoxon tests with Holm correction.
5. Integrated Gradients baseline study (black, white, Gaussian noise, Gaussian blur): attribution statistics **and** faithfulness.
6. Runtime per explanation for each method.
7. Qualitative XAI figures (each model explained with its own Grad-CAM / LIME / IG).
8. `Data/` folder with 40 sample images, `sample_records.csv` and a data dictionary (for the project submission).

Run the cells top to bottom on a CUDA GPU. Expect roughly 1–2 hours in total (LIME is the slowest part).

In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────
import os, json, time, random, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler, ConcatDataset, Subset
from torchvision import datasets, models, transforms
from torchvision.transforms.functional import gaussian_blur

from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve)
from scipy.stats import wilcoxon

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from lime import lime_image
from captum.attr import IntegratedGradients

warnings.filterwarnings('ignore', category=UserWarning)
trapz = getattr(np, 'trapezoid', None) or np.trapz   # numpy 1.x / 2.x

In [ ]:
# ── 2. Configuration ──────────────────────────────────────────────────────
DATA_ROOT      = 'chest_xray'     # folder containing train/ val/ test/
SEED           = 42
VAL_FRACTION   = 0.10             # stratified split carved out of train+val
BATCH_SIZE     = 32
NUM_WORKERS    = 2

STAGE1_EPOCHS, STAGE1_LR = 5, 1e-3    # head only
STAGE2_EPOCHS, STAGE2_LR = 20, 1e-5   # full fine-tune, cosine annealing
PATIENCE       = 5                    # early stopping on validation loss
RUN_TRAINING   = True                 # False = load saved weights from results/models/

N_EVAL          = None   # test images for faithfulness; None = all 624
LIME_SAMPLES    = 1000
IG_STEPS        = 300
IG_INTERNAL_BS  = 50     # lower this if the GPU runs out of memory
FAITH_STEPS     = 50     # deletion/insertion curve resolution
BLUR_KERNEL, BLUR_SIGMA = 51, 15.0    # used for blur baseline and deletion fill
N_BASELINE_IMGS = 100    # images for the IG baseline study
SPARSITY_FRAC   = 0.10   # pixel counts as 'near zero' if |a| < 10 % of max |a|

OUT       = 'results'
MODEL_DIR = os.path.join(OUT, 'models')
FIG_DIR   = os.path.join(OUT, 'figures')
for d in (OUT, MODEL_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'
print('Device:', device, '| torch', torch.__version__)

In [ ]:
# ── 3. Locate the Kaggle dataset (download with kagglehub if missing) ─────
def find_root(start):
    for root, dirs, _ in os.walk(start):
        if {'train', 'test', 'val'} <= set(dirs) and '__MACOSX' not in root:
            return root
    return None

if find_root(DATA_ROOT) is None:
    import kagglehub   # pip install kagglehub
    dl = kagglehub.dataset_download('paultimothymooney/chest-xray-pneumonia')
    DATA_ROOT = dl
DATA_ROOT = find_root(DATA_ROOT)
assert DATA_ROOT is not None, 'Could not find train/ val/ test/ folders'
print('Using data from:', os.path.abspath(DATA_ROOT))
for split in ('train', 'val', 'test'):
    for c in ('NORMAL', 'PNEUMONIA'):
        n = len(os.listdir(os.path.join(DATA_ROOT, split, c)))
        print(f'{split:5s} {c:9s} {n}')

In [ ]:
# ── 4. Transforms, proper validation split, loaders ───────────────────────
# The original Kaggle 'val' folder has only 16 images, which is far too small for
# model selection. We merge train + val and carve out a stratified 10 % validation set.
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
transform_eval = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Built only from torchvision classes so DataLoader workers also work on Windows
train_f = datasets.ImageFolder(os.path.join(DATA_ROOT, 'train'))
val_f   = datasets.ImageFolder(os.path.join(DATA_ROOT, 'val'))
classes = train_f.classes                                   # ['NORMAL', 'PNEUMONIA']
pool    = train_f.samples + val_f.samples
labels  = np.array([y for _, y in pool])
tr_idx, va_idx = train_test_split(np.arange(len(pool)), test_size=VAL_FRACTION,
                                  stratify=labels, random_state=SEED)
train_samples = [pool[i] for i in tr_idx]
val_samples   = [pool[i] for i in va_idx]

pd.DataFrame([(p, classes[y], 'train') for p, y in train_samples] +
             [(p, classes[y], 'val') for p, y in val_samples],
             columns=['path', 'label', 'split']).to_csv(os.path.join(OUT, 'split.csv'), index=False)

def pooled(tf):   # train + original val folders, same order as `pool`
    return ConcatDataset([datasets.ImageFolder(os.path.join(DATA_ROOT, 'train'), tf),
                          datasets.ImageFolder(os.path.join(DATA_ROOT, 'val'), tf)])
train_ds = Subset(pooled(transform_train), tr_idx.tolist())
val_ds   = Subset(pooled(transform_eval), va_idx.tolist())
test_ds  = datasets.ImageFolder(os.path.join(DATA_ROOT, 'test'), transform_eval)

# Class imbalance: oversample the minority class with a WeightedRandomSampler
tr_labels     = np.array([y for _, y in train_samples])
class_w       = 1.0 / np.bincount(tr_labels)
sampler       = WeightedRandomSampler(torch.as_tensor(class_w[tr_labels], dtype=torch.double),
                                      num_samples=len(tr_labels), replacement=True,
                                      generator=torch.Generator().manual_seed(SEED))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train {len(train_ds)} {np.bincount(tr_labels)} | '
      f'Val {len(val_ds)} {np.bincount([y for _, y in val_samples])} | '
      f'Test {len(test_ds)} {np.bincount(test_ds.targets)} | classes {classes}')

In [ ]:
# ── 5. Models ─────────────────────────────────────────────────────────────
def build_resnet50(pretrained=True):
    m = models.resnet50(weights='IMAGENET1K_V1' if pretrained else None)
    m.fc = nn.Sequential(nn.Linear(m.fc.in_features, 512), nn.ReLU(),
                         nn.Dropout(0.3), nn.Linear(512, 2))
    return m

class DenseNetXAI(nn.Module):
    # Same layers/state_dict keys as torchvision DenseNet121, but the final ReLU is
    # not in-place, which Grad-CAM and Captum need.
    def __init__(self, pretrained=True):
        super().__init__()
        base = models.densenet121(weights='IMAGENET1K_V1' if pretrained else None)
        self.features   = base.features
        self.classifier = nn.Linear(base.classifier.in_features, 2)
    def forward(self, x):
        out = F.relu(self.features(x), inplace=False)
        out = F.adaptive_avg_pool2d(out, 1).flatten(1)
        return self.classifier(out)

MODELS = {
    'ResNet50':    dict(build=build_resnet50, head=lambda m: m.fc,
                        cam_layer=lambda m: m.layer4[-1]),
    'DenseNet121': dict(build=DenseNetXAI, head=lambda m: m.classifier,
                        cam_layer=lambda m: m.features.denseblock4),
}

In [ ]:
# ── 6. Two-stage training with early stopping on validation loss ──────────
criterion = nn.CrossEntropyLoss()

def train_one_epoch(model, opt, scaler):
    model.train(); total, n = 0.0, 0
    for x, y in train_loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device.type, enabled=USE_AMP):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        total += loss.item() * len(y); n += len(y)
    return total / n

@torch.no_grad()
def predict(model, loader):
    model.eval(); probs, ys, total = [], [], 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.autocast(device.type, enabled=USE_AMP):
            out = model(x)
        total += criterion(out.float(), y).item() * len(y)
        probs.append(torch.softmax(out.float(), 1)[:, 1].cpu()); ys.append(y.cpu())
    probs, ys = torch.cat(probs).numpy(), torch.cat(ys).numpy()
    return total / len(ys), probs, ys

def train_two_stage(name):
    set_seed()
    spec  = MODELS[name]
    model = spec['build']().to(device)
    path  = os.path.join(MODEL_DIR, f'{name}.pth')
    scaler, hist = torch.amp.GradScaler(enabled=USE_AMP), []

    # Stage 1: head only
    for p in model.parameters(): p.requires_grad = False
    for p in spec['head'](model).parameters(): p.requires_grad = True
    opt = torch.optim.Adam(spec['head'](model).parameters(), lr=STAGE1_LR)
    for ep in range(STAGE1_EPOCHS):
        tl = train_one_epoch(model, opt, scaler)
        vl, vp, vy = predict(model, val_loader)
        hist.append(dict(stage=1, epoch=ep + 1, train_loss=tl, val_loss=vl,
                         val_acc=((vp > 0.5) == vy).mean(), val_auc=roc_auc_score(vy, vp)))
        print(f'[{name}] S1 {ep+1}/{STAGE1_EPOCHS} train {tl:.4f} val {vl:.4f} acc {hist[-1]["val_acc"]:.4f}')

    # Stage 2: full fine-tune with cosine annealing + early stopping
    for p in model.parameters(): p.requires_grad = True
    opt   = torch.optim.Adam(model.parameters(), lr=STAGE2_LR)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=STAGE2_EPOCHS)
    best, wait = float('inf'), 0
    for ep in range(STAGE2_EPOCHS):
        tl = train_one_epoch(model, opt, scaler)
        vl, vp, vy = predict(model, val_loader)
        sched.step()
        hist.append(dict(stage=2, epoch=STAGE1_EPOCHS + ep + 1, train_loss=tl, val_loss=vl,
                         val_acc=((vp > 0.5) == vy).mean(), val_auc=roc_auc_score(vy, vp)))
        flag = ''
        if vl < best:
            best, wait, flag = vl, 0, '  * saved'
            torch.save(model.state_dict(), path)
        else:
            wait += 1
        print(f'[{name}] S2 {ep+1}/{STAGE2_EPOCHS} train {tl:.4f} val {vl:.4f} '
              f'acc {hist[-1]["val_acc"]:.4f} auc {hist[-1]["val_auc"]:.4f}{flag}')
        if wait >= PATIENCE:
            print('Early stopping.'); break
    pd.DataFrame(hist).to_csv(os.path.join(OUT, f'history_{name}.csv'), index=False)
    model.load_state_dict(torch.load(path, map_location=device))
    return model

def load_model(name):
    m = MODELS[name]['build'](pretrained=False)
    m.load_state_dict(torch.load(os.path.join(MODEL_DIR, f'{name}.pth'), map_location=device))
    return m.to(device)

trained = {}
for name in MODELS:
    trained[name] = train_two_stage(name) if RUN_TRAINING else load_model(name)
for m in trained.values():
    m.eval()
    for p in m.parameters(): p.requires_grad = True   # needed for gradient-based XAI

In [ ]:
# ── 7. Test-set evaluation ────────────────────────────────────────────────
def youden_threshold(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    return float(thr[np.argmax(tpr - fpr)])

def metrics_at(y, p, thr):
    pred = (p >= thr).astype(int)
    rep  = classification_report(y, pred, target_names=classes, output_dict=True, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    return dict(threshold=thr, accuracy=rep['accuracy'],
                normal_recall=rep['NORMAL']['recall'], pneumonia_recall=rep['PNEUMONIA']['recall'],
                normal_precision=rep['NORMAL']['precision'], pneumonia_precision=rep['PNEUMONIA']['precision'],
                pneumonia_f1=rep['PNEUMONIA']['f1-score'], auc_roc=roc_auc_score(y, p),
                TN=int(tn), FP=int(fp), FN=int(fn), TP=int(tp))

rows, test_probs = [], {}
for name, model in trained.items():
    _, vp, vy = predict(model, val_loader)
    _, tp_, ty = predict(model, test_loader)
    test_probs[name] = tp_
    rows.append(dict(model=name, setting='threshold 0.5', **metrics_at(ty, tp_, 0.5)))
    rows.append(dict(model=name, setting='val-tuned (Youden)', **metrics_at(ty, tp_, youden_threshold(vy, vp))))
cls_df = pd.DataFrame(rows)
cls_df.to_csv(os.path.join(OUT, 'classification_metrics.csv'), index=False)
test_labels = ty
display(cls_df.round(4))

# Confusion matrices, ROC and learning curves
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, (name, p) in zip(axes, test_probs.items()):
    cm = confusion_matrix(test_labels, (p >= 0.5).astype(int))
    ax.imshow(cm, cmap='Blues')
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=14)
    ax.set_xticks([0, 1], classes); ax.set_yticks([0, 1], classes)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title(name)
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, 'confusion_matrices.png'), dpi=300); plt.show()

plt.figure(figsize=(5, 4))
for name, p in test_probs.items():
    fpr, tpr, _ = roc_curve(test_labels, p)
    plt.plot(fpr, tpr, label=f'{name} (AUC {roc_auc_score(test_labels, p):.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=0.8)
plt.xlabel('False positive rate'); plt.ylabel('True positive rate'); plt.legend(); plt.title('ROC curves')
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, 'roc_curves.png'), dpi=300); plt.show()

if RUN_TRAINING:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    for name in trained:
        h = pd.read_csv(os.path.join(OUT, f'history_{name}.csv'))
        axes[0].plot(h.epoch, h.train_loss, '--', label=f'{name} train')
        axes[0].plot(h.epoch, h.val_loss, label=f'{name} val')
        axes[1].plot(h.epoch, h.val_auc, label=name)
    axes[0].axvline(STAGE1_EPOCHS + 0.5, color='grey', lw=0.8)
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend(fontsize=8)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Validation AUC'); axes[1].legend(fontsize=8)
    plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, 'learning_curves.png'), dpi=300); plt.show()

In [ ]:
# ── 8. XAI methods and faithfulness metric ────────────────────────────────
MEAN_T = torch.tensor(MEAN, device=device).view(1, 3, 1, 1)
STD_T  = torch.tensor(STD, device=device).view(1, 3, 1, 1)
to01   = lambda x: (x * STD_T + MEAN_T).clamp(0, 1)      # normalized -> [0,1]
from01 = lambda x: (x - MEAN_T) / STD_T                   # [0,1] -> normalized
blur   = lambda x: gaussian_blur(x, [BLUR_KERNEL, BLUR_KERNEL], [BLUR_SIGMA, BLUR_SIGMA])

def sync():
    if device.type == 'cuda': torch.cuda.synchronize()

@torch.no_grad()
def probs_batched(model, imgs, chunk=64):
    out = []
    for i in range(0, len(imgs), chunk):
        with torch.autocast(device.type, enabled=USE_AMP):
            out.append(torch.softmax(model(imgs[i:i + chunk]).float(), 1))
    return torch.cat(out)

# Grad-CAM (hooks are released after every call so they never accumulate)
def gradcam_map(model, layer, x, target):
    cam = GradCAM(model=model, target_layers=[layer])
    try:
        return cam(input_tensor=x, targets=[ClassifierOutputTarget(target)])[0]
    finally:
        cam.activations_and_grads.release()

# LIME: continuous saliency = surrogate weight of each superpixel
lime_explainer = lime_image.LimeImageExplainer(random_state=SEED)
def lime_map(model, x, target, num_samples=LIME_SAMPLES):
    img = to01(x)[0].permute(1, 2, 0).cpu().numpy().astype(np.double)
    def predict_fn(batch):
        t = torch.from_numpy(batch).permute(0, 3, 1, 2).float().to(device)
        return probs_batched(model, from01(t)).cpu().numpy()
    exp = lime_explainer.explain_instance(img, predict_fn, labels=(target,), top_labels=None,
                                          hide_color=0, num_samples=num_samples, batch_size=100,
                                          random_seed=SEED, progress_bar=False)
    sal = np.zeros(exp.segments.shape, dtype=np.float32)
    for seg_id, w in exp.local_exp[target]:
        sal[exp.segments == seg_id] = w
    return sal

# Integrated Gradients with selectable baseline (all baselines built in pixel space)
def make_baseline(x, kind):
    if kind == 'black': return from01(torch.zeros_like(x))
    if kind == 'white': return from01(torch.ones_like(x))
    if kind == 'noise':
        g = torch.Generator(device=x.device).manual_seed(SEED)
        return from01((0.5 + 0.2 * torch.randn(x.shape, generator=g, device=x.device)).clamp(0, 1))
    if kind == 'blur':  return blur(x)
    raise ValueError(kind)

def ig_attr(model, x, target, kind='black', steps=IG_STEPS):
    ig = IntegratedGradients(model)
    a, delta = ig.attribute(x, baselines=make_baseline(x, kind), target=target, n_steps=steps,
                            internal_batch_size=IG_INTERNAL_BS, return_convergence_delta=True)
    return a[0].sum(0).detach().cpu().numpy(), float(delta.abs().item())

# Deletion / insertion (Petsiuk et al.): pixels ranked by saliency are replaced by
# (deletion) or revealed from (insertion) a blurred copy of the image.
def deletion_insertion(model, x, sal, target, steps=FAITH_STEPS):
    H, W = sal.shape; n = H * W
    ranks = np.empty(n, dtype=np.int64)
    ranks[np.argsort(-sal.ravel(), kind='stable')] = np.arange(n)
    ranks = torch.from_numpy(ranks).view(1, 1, H, W).to(device)
    k     = torch.linspace(0, n, steps + 1, device=device).round().long().view(-1, 1, 1, 1)
    mask  = (ranks < k).float()                     # 1 = pixel already removed / revealed
    fill  = blur(x)
    d = probs_batched(model, x * (1 - mask) + fill * mask)[:, target].cpu().numpy()
    i = probs_batched(model, fill * (1 - mask) + x * mask)[:, target].cpu().numpy()
    return float(trapz(d, dx=1 / steps)), float(trapz(i, dx=1 / steps)), d, i

In [ ]:
# ── 9. Faithfulness on the test set: 2 models x (Grad-CAM, LIME, IG, random) ─
# Explanations target the model's predicted class. Results are written after every
# image, so if the run stops you can simply re-run this cell and it resumes.
if N_EVAL is None:
    eval_idx = list(range(len(test_ds)))
else:
    eval_idx, _ = train_test_split(np.arange(len(test_ds)), train_size=N_EVAL,
                                   stratify=test_ds.targets, random_state=SEED)
    eval_idx = sorted(eval_idx.tolist())

FAITH_CSV, CURVE_NPZ = os.path.join(OUT, 'faithfulness_per_image.csv'), os.path.join(OUT, 'curves.npz')
done  = pd.read_csv(FAITH_CSV) if os.path.exists(FAITH_CSV) else pd.DataFrame()
curves = dict(np.load(CURVE_NPZ)) if os.path.exists(CURVE_NPZ) else {}
done_keys = set(zip(done.get('model', []), done.get('img_idx', [])))
rng = np.random.default_rng(SEED)

for name, model in trained.items():
    layer = MODELS[name]['cam_layer'](model)
    for idx in tqdm(eval_idx, desc=name):
        if (name, idx) in done_keys: continue
        x, y = test_ds[idx]; x = x.unsqueeze(0).to(device)
        p = probs_batched(model, x)[0]; target = int(p.argmax())
        rows = []
        for method in ('Grad-CAM', 'LIME', 'IG', 'Random'):
            sync(); t0 = time.perf_counter()
            if method == 'Grad-CAM': sal = gradcam_map(model, layer, x, target)
            elif method == 'LIME':   sal = lime_map(model, x, target)
            elif method == 'IG':     sal, _ = ig_attr(model, x, target, 'black')
            else:                    sal = rng.random((224, 224)).astype(np.float32)
            sync(); secs = time.perf_counter() - t0
            d_auc, i_auc, d, i = deletion_insertion(model, x, sal, target)
            rows.append(dict(model=name, method=method, img_idx=idx, path=test_ds.samples[idx][0],
                             label=classes[y], pred=classes[target], prob=float(p[target]),
                             correct=int(y == target), deletion_auc=d_auc, insertion_auc=i_auc,
                             seconds=secs))
            curves[f'{name}|{method}|{idx}|del'] = d; curves[f'{name}|{method}|{idx}|ins'] = i
        done = pd.concat([done, pd.DataFrame(rows)], ignore_index=True)
        done.to_csv(FAITH_CSV, index=False)
        if len(done) % 200 == 0: np.savez_compressed(CURVE_NPZ, **curves)
np.savez_compressed(CURVE_NPZ, **curves)
faith = done
print(faith.groupby(['model', 'method'])[['deletion_auc', 'insertion_auc', 'seconds']].mean().round(4))

In [ ]:
# ── 10. Statistics: mean ± SE, bootstrap 95 % CI, paired Wilcoxon (Holm) ──
def boot_ci(v, n=5000, seed=SEED):
    r = np.random.default_rng(seed)
    means = r.choice(v, (n, len(v)), replace=True).mean(1)
    return np.percentile(means, [2.5, 97.5])

summ = []
for (m, meth), g in faith.groupby(['model', 'method']):
    row = dict(model=m, method=meth, n=len(g))
    for col in ('deletion_auc', 'insertion_auc'):
        v = g[col].values; lo, hi = boot_ci(v)
        row.update({f'{col}_mean': v.mean(), f'{col}_se': v.std(ddof=1) / np.sqrt(len(v)),
                    f'{col}_ci_low': lo, f'{col}_ci_high': hi})
    row['seconds_mean'] = g.seconds.mean()
    summ.append(row)
summary = pd.DataFrame(summ)
gc_time = summary.set_index(['model', 'method'])['seconds_mean']
summary['x_slower_than_gradcam'] = [gc_time[(m, me)] / gc_time[(m, 'Grad-CAM')]
                                    for m, me in zip(summary.model, summary.method)]
summary.to_csv(os.path.join(OUT, 'faithfulness_summary.csv'), index=False)
display(summary.round(4))

def holm(pvals):
    order = np.argsort(pvals); adj = np.empty(len(pvals)); running = 0
    for rank, i in enumerate(order):
        running = max(running, (len(pvals) - rank) * pvals[i]); adj[i] = min(1, running)
    return adj

tests = []
pairs = [('IG', 'Grad-CAM'), ('IG', 'LIME'), ('LIME', 'Grad-CAM'),
         ('Grad-CAM', 'Random'), ('LIME', 'Random'), ('IG', 'Random')]
for m in faith.model.unique():
    w = faith[faith.model == m].pivot(index='img_idx', columns='method')
    for col in ('deletion_auc', 'insertion_auc'):
        for a, b in pairs:
            diff = w[col][a] - w[col][b]
            tests.append(dict(comparison=f'{m}: {a} vs {b}', metric=col,
                              median_diff=diff.median(), p=wilcoxon(w[col][a], w[col][b]).pvalue))
for meth in ('Grad-CAM', 'LIME', 'IG'):
    w = faith[faith.method == meth].pivot(index='img_idx', columns='model')
    for col in ('deletion_auc', 'insertion_auc'):
        diff = w[col]['DenseNet121'] - w[col]['ResNet50']
        tests.append(dict(comparison=f'{meth}: DenseNet121 vs ResNet50', metric=col,
                          median_diff=diff.median(),
                          p=wilcoxon(w[col]['DenseNet121'], w[col]['ResNet50']).pvalue))
tests = pd.DataFrame(tests)
tests['p_holm'] = holm(tests.p.values)
tests['significant_0.05'] = tests.p_holm < 0.05
tests.to_csv(os.path.join(OUT, 'wilcoxon_tests.csv'), index=False)
display(tests.round(5))

In [ ]:
# ── 11. Faithfulness figures ──────────────────────────────────────────────
methods = ['Grad-CAM', 'LIME', 'IG', 'Random']
colors  = {'Grad-CAM': '#2a78d6', 'LIME': '#eb6834', 'IG': '#1baf7a', 'Random': '#9a9a9a'}
xs = np.linspace(0, 1, FAITH_STEPS + 1)
fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True)
for r, name in enumerate(trained):
    for c, kind in enumerate(('del', 'ins')):
        ax = axes[r, c]
        for meth in methods:
            arr = np.stack([v for k, v in curves.items()
                            if k.startswith(f'{name}|{meth}|') and k.endswith(kind)])
            ax.plot(xs, arr.mean(0), color=colors[meth], label=meth,
                    ls='--' if meth == 'Random' else '-')
        ax.set_title(f'{name} — {"Deletion" if kind == "del" else "Insertion"}')
        ax.set_ylabel('Probability of explained class')
axes[1, 0].set_xlabel('Fraction of pixels removed'); axes[1, 1].set_xlabel('Fraction of pixels inserted')
axes[0, 0].legend()
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, 'faithfulness_curves.png'), dpi=300); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, col, title in [(axes[0], 'deletion_auc', 'Deletion AUC (lower is better)'),
                       (axes[1], 'insertion_auc', 'Insertion AUC (higher is better)')]:
    for j, name in enumerate(trained):
        s = summary[summary.model == name].set_index('method').loc[methods]
        xpos = np.arange(len(methods)) + (j - 0.5) * 0.38
        err = [s[f'{col}_mean'] - s[f'{col}_ci_low'], s[f'{col}_ci_high'] - s[f'{col}_mean']]
        ax.bar(xpos, s[f'{col}_mean'], 0.36, yerr=err, capsize=3, label=name)
    ax.set_xticks(range(len(methods)), methods); ax.set_title(title)
axes[0].legend()
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, 'faithfulness_bars.png'), dpi=300); plt.show()

In [ ]:
# ── 12. Integrated Gradients baseline study ───────────────────────────────
# Same N_BASELINE_IMGS images for every baseline and model, so results are paired.
base_idx = eval_idx[:: max(1, len(eval_idx) // N_BASELINE_IMGS)][:N_BASELINE_IMGS]
rows = []
for name, model in trained.items():
    for idx in tqdm(base_idx, desc=f'IG baselines {name}'):
        x, _ = test_ds[idx]; x = x.unsqueeze(0).to(device)
        target = int(probs_batched(model, x)[0].argmax())
        for kind in ('black', 'white', 'noise', 'blur'):
            a, delta = ig_attr(model, x, target, kind)
            aa = np.abs(a)
            d_auc, i_auc, _, _ = deletion_insertion(model, x, a, target)
            rows.append(dict(model=name, baseline=kind, img_idx=idx,
                             mean_abs=aa.mean(), max_abs=aa.max(), std_abs=aa.std(),
                             sparsity_pct=100 * (aa < SPARSITY_FRAC * aa.max()).mean(),
                             completeness_err=delta, deletion_auc=d_auc, insertion_auc=i_auc))
ig_base = pd.DataFrame(rows)
ig_base.to_csv(os.path.join(OUT, 'ig_baselines_per_image.csv'), index=False)
ig_summary = ig_base.groupby(['model', 'baseline']).agg(
    mean_abs=('mean_abs', 'mean'), max_abs=('max_abs', 'mean'), std_abs=('std_abs', 'mean'),
    sparsity_pct=('sparsity_pct', 'mean'), completeness_err=('completeness_err', 'mean'),
    deletion_auc=('deletion_auc', 'mean'), insertion_auc=('insertion_auc', 'mean')).reset_index()
ig_summary.to_csv(os.path.join(OUT, 'ig_baselines_summary.csv'), index=False)
display(ig_summary.round(4))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
order = ['black', 'white', 'noise', 'blur']
for ax, col in zip(axes, ['sparsity_pct', 'deletion_auc', 'insertion_auc']):
    for j, name in enumerate(trained):
        s = ig_summary[ig_summary.model == name].set_index('baseline').loc[order]
        ax.bar(np.arange(4) + (j - 0.5) * 0.38, s[col], 0.36, label=name)
    ax.set_xticks(range(4), order); ax.set_title(col)
axes[0].legend()
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, 'ig_baselines.png'), dpi=300); plt.show()

In [ ]:
# ── 13. Qualitative figures (each model explained with its OWN explanations) ─
from skimage.segmentation import mark_boundaries
pneu = [i for i in eval_idx if test_ds.targets[i] == 1][:3]
for name, model in trained.items():
    layer = MODELS[name]['cam_layer'](model)
    fig, axes = plt.subplots(len(pneu), 4, figsize=(14, 3.5 * len(pneu)))
    for r, idx in enumerate(pneu):
        x, _ = test_ds[idx]; x = x.unsqueeze(0).to(device)
        target = int(probs_batched(model, x)[0].argmax())
        img = to01(x)[0].permute(1, 2, 0).cpu().numpy()
        gc  = gradcam_map(model, layer, x, target)
        lm  = lime_map(model, x, target)
        ig, _ = ig_attr(model, x, target, 'black')
        top = lm >= np.sort(lm.ravel())[::-1][min(len(lm.ravel()) - 1, 224 * 224 // 10)]
        panels = [(img, 'Original'), (gc, 'Grad-CAM'), (mark_boundaries(img, top.astype(int)), 'LIME'),
                  (np.abs(ig), 'Integrated Gradients')]
        for c, (im, title) in enumerate(panels):
            ax = axes[r, c]; ax.axis('off')
            if title in ('Grad-CAM', 'Integrated Gradients'):
                ax.imshow(img); ax.imshow(im / (im.max() + 1e-8), cmap='jet', alpha=0.45)
            else:
                ax.imshow(im)
            if r == 0: ax.set_title(title)
    plt.suptitle(f'{name} — pneumonia test cases'); plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f'xai_examples_{name}.png'), dpi=300); plt.show()

In [ ]:
# ── 14. Sample data for the submission ZIP (Data/ folder) ─────────────────
SAMPLE_DIR = os.path.join('Data', 'sample_images'); os.makedirs(SAMPLE_DIR, exist_ok=True)
r = np.random.default_rng(SEED); recs = []
for cls_id, cls in enumerate(classes):
    pick = r.choice([i for i, t in enumerate(test_ds.targets) if t == cls_id], 20, replace=False)
    for idx in sorted(pick):
        src = test_ds.samples[idx][0]; fn = os.path.basename(src)
        shutil.copy(src, os.path.join(SAMPLE_DIR, fn))
        im = Image.open(src)
        g  = np.asarray(im.convert('L'), dtype=np.float32)
        recs.append(dict(file_name=fn, split='test', true_label=cls,
                         width_px=im.width, height_px=im.height, color_mode=im.mode,
                         mean_intensity=round(float(g.mean()), 2), std_intensity=round(float(g.std()), 2),
                         resnet50_prob_pneumonia=round(float(test_probs['ResNet50'][idx]), 4),
                         densenet121_prob_pneumonia=round(float(test_probs['DenseNet121'][idx]), 4),
                         resnet50_pred=classes[int(test_probs['ResNet50'][idx] >= 0.5)],
                         densenet121_pred=classes[int(test_probs['DenseNet121'][idx] >= 0.5)]))
pd.DataFrame(recs).to_csv(os.path.join('Data', 'sample_records.csv'), index=False)

dictionary = '''# Data dictionary — sample_records.csv

| Field | Type | Description |
|---|---|---|
| file_name | text | Image file name inside `sample_images/` (original Kaggle file name) |
| split | text | Dataset split the image comes from (all samples are from the held-out test set) |
| true_label | text | Ground-truth class: NORMAL or PNEUMONIA |
| width_px, height_px | integer | Original image size in pixels (resized to 224x224 for the models) |
| color_mode | text | PIL colour mode of the original file (L = greyscale, RGB = colour) |
| mean_intensity | float | Mean greyscale pixel value, 0-255 |
| std_intensity | float | Standard deviation of greyscale pixel values, 0-255 |
| resnet50_prob_pneumonia | float | ResNet50 predicted probability of PNEUMONIA (0-1) |
| densenet121_prob_pneumonia | float | DenseNet121 predicted probability of PNEUMONIA (0-1) |
| resnet50_pred, densenet121_pred | text | Predicted class at threshold 0.5 |

Source: Kermany et al., Chest X-Ray Images (Pneumonia), Kaggle — paultimothymooney/chest-xray-pneumonia.
Full dataset: 5,856 pediatric chest X-rays (4,273 pneumonia, 1,583 normal).
'''
open(os.path.join('Data', 'data_dictionary.md'), 'w').write(dictionary)
print('Saved', len(recs), 'sample records to Data/')

In [ ]:
# ── 15. One-page summary of every number the paper needs ─────────────────
report = dict(
    split=dict(train=len(train_ds), val=len(val_ds), test=len(test_ds), seed=SEED),
    classification=cls_df.to_dict('records'),
    faithfulness=summary.to_dict('records'),
    wilcoxon=tests.to_dict('records'),
    ig_baselines=ig_summary.to_dict('records'),
    config=dict(lime_samples=LIME_SAMPLES, ig_steps=IG_STEPS, faith_steps=FAITH_STEPS,
                n_eval=len(eval_idx), n_baseline_imgs=len(base_idx),
                stage1=(STAGE1_EPOCHS, STAGE1_LR), stage2=(STAGE2_EPOCHS, STAGE2_LR), patience=PATIENCE),
)
with open(os.path.join(OUT, 'all_results.json'), 'w') as f:
    json.dump(report, f, indent=2, default=float)
print('Done. Send results/all_results.json and results/figures/ to update the paper.')